<p style="padding: 10px; border: 1px solid black;">
<img src="../mlu_utils/images/MLU-NEW-logo.png" alt="drawing" width="400"/> <br/>
<div style="background-color: #1A5276; padding: 20px; border-radius: 10px; text-align: center; margin-bottom: 30px;">
    <h1 style="color: white; margin: 0;">MLU: Building Applications with Foundation Models</h1>
    <h2 style="color: white; margin-top: 15px;">Lab 3b: Multimodal Retrieval Augmented Generation</h2>
</div>

<!-- Table of Contents with All Section Levels -->
<div style="background-color: #EBF5FB; padding: 15px; border-radius: 5px; margin-bottom: 30px;">
    <h2 style="color: #2874A6; border-bottom: 1px solid #2874A6; padding-bottom: 5px;">Table of Contents</h2>
    <p><a href="#section1" style="color: #2E86C1; font-weight: bold; text-decoration: none;">1. Installing dependencies</a></p>
    <p><a href="#section2" style="color: #2E86C1; font-weight: bold; text-decoration: none;">2. Process PDF</a></p>
    <p><a href="#section3" style="color: #2E86C1; font-weight: bold; text-decoration: none;">3. Generate Multimodal Embeddings</a></p>
    <p><a href="#section4" style="color: #2E86C1; font-weight: bold; text-decoration: none;">4. Create vector database</a></p>
    <p><a href="#section5" style="color: #2E86C1; font-weight: bold; text-decoration: none;">5. Generate a RAG Response</a></p>
    <p><a href="#section6" style="color: #2E86C1; font-weight: bold; text-decoration: none;">6. Test RAG Workflow</a></p>
    <p><a href="#section7" style="color: #2E86C1; font-weight: bold; text-decoration: none;">7. Quizzes</a></p>
</div>

This exercise shows how to implement a multimodal retrieval augmented generation (RAG) system. In retrieval augmented generation, an external source of information as well as the input prompt are used to generate the response. In a multimodal setting, one of the most popular use cases is to include the images in the response generation process. 

This exercise implements the multimodal RAG system by using a PDF file that include images, text, and tables. This PDF file is the external source of information mentioned earlier in RAG definition. Once the system is setup, the model will be able to generate its responses considering the images, text, and tables from the PDF provided.

<!-- Compact Lab Introduction with Activity/Challenge Explanation -->
<div style="background-color: #F8F9F9; padding: 15px; border-radius: 5px; margin: 20px 0;">
    <h4 style="color: #2E4053; margin-top: 0;">About This Lab</h4>
    <p>Throughout this lab, you will encounter two types of interactive elements:</p>
    <table style="width: 100%; border-collapse: collapse; margin: 15px 0;">
        <tr>
            <td style="text-align: center; padding: 10px; width: 50%;">
                <img src="../mlu_utils/images/activity.png" alt="Activity" width="125"/>
            </td>
            <td style="text-align: center; padding: 10px; width: 50%;">
                <img src="../mlu_utils/images/challenge.png" alt="Challenge" width="125"/>
            </td>
        </tr>
        <tr>
            <td style="text-align: center; padding: 10px; background-color: #EBF5FB;">
                <p>No coding is needed for an activity. You try to understand a concept, <br/>answer questions, or run a code cell.</p>
            </td>
            <td style="text-align: center; padding: 10px; background-color: #FEF9E7;">
                <p>Challenges are where you test your understanding by implementing something new or taking a short quiz.</p>
            </td>
        </tr>
    </table>
    <p>Please work through this notebook from top to bottom to avoid errors due to missing code or context.</p>
</div>

<!-- Section Header -->
<div id="section1" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">1. Installing dependencies</h2>
</div>

__Note:__ the pip command below will output some error messages. You can disregard them as they are not affecting the notebook.

Installing the required libraries:

In [ ]:
%%capture
!pip install -q -r ../requirements.txt

Importing the libraries used in this exercise:

In [ ]:
import sys
sys.path.append('..')

import boto3
from botocore.exceptions import ClientError
import os
import json
import numpy as np
import base64
import pymupdf
import pandas as pd
from PIL import Image
import faiss
from tqdm import tqdm
from IPython import display

# Import utility functions that provide answers to challenges
%load_ext autoreload
%aimport mlu_utils.course_utils

<!-- Section Header -->
<div id="section2" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">2. Process PDF</h2>
</div>

In this lab, we will read the sample PDF file of a the well-known paper __"Attention Is All You Need"__ paper by Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin. This research paper laid out the foundations of the transformers models that power many generative AI applications nowadays. Paper linked [here](https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf). The paper is 11 pages long.

<!-- Challenge Box -->
<div style="background-color: #FEF9E7; border-left: 5px solid #F1C40F; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../mlu_utils/images/challenge.png" alt="Challenge" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #B7950B; margin-top: 0;">Challenge: Test different documents</h4>
        <p>After observing how the RAG workflow is implemented in this lab, test it with other PDFs and observe how well it performds with varying document formats.</p>
    </div>
</div>

In [ ]:
filename = "transformers_paper.pdf"

display.IFrame("data/"+ filename, width=600, height=600)

### Extract text and images from each page

The contents of the PDF need to be extracted and processed to be compatible with the RAG application.

The following are the steps we will follow to process the data in this section:
1. Extract the data (text and images) from the PDF using pymupdf.
2. Go through each page. Create smaller text chunks from the text of the page.
3. Convert each page of the PDF into an image.
3. For each text chunk, image and page, generate embeddings using Amazon Nova 2 Multimodal Embeddings.  
4. Save the information of each page in a list to store in a vector database.

In [ ]:
from mlu_utils.multimodal_utils import pdf2imgs

doc = pymupdf.open("data/"+ filename)
num_pages = len(doc)

# Define the directories to store the extracted text, images and page images from each page
image_save_dir = "images/data"
text_save_dir = "text/data"
page_images_save_dir = "page_images"

# Chunk the text for effective retrieval
chunk_size = 700
overlap=200


items = []
# Process all pages of the PDF
for page_num in tqdm(range(num_pages), desc="Processing PDF pages"):
    page = page = doc[page_num]
    text = page.get_text()
    
    # Process chunks with overlap
    chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size-overlap)]
    
    # Generate an item to add to items
    for i,chunk in enumerate(chunks):
        text_file_name = f"{text_save_dir}/{filename}_text_{page_num}_{i}.txt"
        print("text_file_name", text_file_name)
        print("text_save_dir", text_save_dir)
        print("filename", filename)
        # If the text folder doesn't exist, create one
        os.makedirs(text_save_dir, exist_ok=True)
        with open(text_file_name, 'w') as f:
            f.write(chunk)
        
        item={}
        item["page"] = page_num
        item["type"] = "text"
        item["text"] = chunk
        item["path"] = text_file_name
        items.append(item)
    
    
    # Get all the images in the current page
    images = page.get_images()
    for idx, image in enumerate(images):        
        # Extract the image data
        xref = image[0]
        pix = pymupdf.Pixmap(doc, xref)
        pix.tobytes("png")
        # Create the image_name that includes the image path
        image_name = f"{image_save_dir}/{filename}_image_{page_num}_{idx}_{xref}.png"
        # If the image folder doesn't exist, create one
        os.makedirs(image_save_dir, exist_ok=True)
        # Save the image
        pix.save(image_name)
        
        # Produce base64 string
        with open(image_name, 'rb') as f:
            image = base64.b64encode(f.read()).decode('utf8')
        
        item={}
        item["page"] = page_num
        item["type"] = "image"
        item["path"] = image_name
        item["image"] = image
        items.append(item)

# Save pdf pages as images
page_images_save_dir = pdf2imgs("data/" + filename, page_images_save_dir)

for page_num in range(num_pages):
    page_path = os.path.join(page_images_save_dir,  f"page_{page_num:03d}.png")
    
    # Produce base64 string
    with open(image_name, 'rb') as f:
        page_image = base64.b64encode(f.read()).decode('utf8')
    
    item = {}
    item["page"] = page_num
    item["type"] = "page"
    item["path"] = page_path
    item["image"] = page_image
    items.append(item)

<!-- Challenge Box -->
<div style="background-color: #FEF9E7; border-left: 5px solid #F1C40F; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../mlu_utils/images/challenge.png" alt="Challenge" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #B7950B; margin-top: 0;">Challenge: Types of chunking</h4>
        <p>In the cell above, we have used a simple character-based chunking solution. This can result in broken sentences and words, losing a lot of semantic and syntactic meaning. Try updating the chunking process using a different strategy to preserve the structure and meaning of the document. You can use modules offered by LangChain for this purpose.</p>
    </div>
</div>

<!-- Section Header -->
<div id="section3" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">3. Generate Multimodal Embeddings</h2>
</div>

We will use the same function defined in Lab 2 to generate embeddings from text or image data.

The following function is used to generate multimodal embeddings using Amazon's Nova 2 Multimodal Embeddings model. Embeddings can be generated with text data, image data or both.

In [ ]:
def generate_multimodal_embeddings(prompt=None, image=None, output_embedding_length = 1024):
    """
    Invoke the Amazon Nova 2 Multimodal Embeddings model using AWS Bedrock runtime.

    Args:
        prompt (str): The text prompt to provide to the model.
        image (str): A base64-encoded image data.
        output_embedding_length (int): The dimension of the output embedding (default: 1024).
    Returns:
        list: The embedding vector.

    Raises:
        ValueError: If neither prompt nor image is provided.
    """
    if not prompt and not image:
        raise ValueError("Please provide either a text prompt, base64 image or both as input")
    
    # Initialize the Amazon Bedrock runtime client
    client = boto3.client(service_name="bedrock-runtime")
    model_id = "amazon.nova-2-multimodal-embeddings-v1:0"
    
    # Detect image format if image is provided
    image_format = "png"  # Default format
    if image:
        try:
            image_bytes = base64.b64decode(image)
            # Check PNG signature
            if image_bytes[:8] == b'\\x89PNG\\r\\n\\x1a\\n':
                image_format = "png"
            # Check JPEG signature
            elif image_bytes[:2] == b'\\xff\\xd8':
                image_format = "jpeg"
            # Check GIF signature
            elif image_bytes[:6] in (b'GIF87a', b'GIF89a'):
                image_format = "gif"
            # Check WebP signature
            elif image_bytes[:4] == b'RIFF' and image_bytes[8:12] == b'WEBP':
                image_format = "webp"
        except Exception:
            # Default to png if detection fails
            image_format = "png"
    
    # Build the request body based on input type
    body = {
        "taskType": "SINGLE_EMBEDDING",
        "singleEmbeddingParams": {
            "embeddingDimension": output_embedding_length,
            "embeddingPurpose": "GENERIC_INDEX"
        }
    }
    
    if prompt:
        body["singleEmbeddingParams"]["text"] = {
            "truncationMode": "END",
            "value": prompt
        }
    
    if image:
        body["singleEmbeddingParams"]["image"] = {
            "format": image_format,
            "source": {"bytes": image}
        }

    try:
        response = client.invoke_model(
            modelId=model_id,
            body=json.dumps(body)
        )

        # Process and return the response
        result = json.loads(response.get("body").read())
        return result["embeddings"][0]["embedding"]

    except ClientError as err:
        print(
            f"Couldn't invoke Nova embedding model. Here's why: {err.response['Error']['Code']}: {err.response['Error']['Message']}"
        )
        raise

#### Let's use the `generate_multimodal_embeddings` function to generate embeddings of every item extracted from the PDF

In [ ]:
embedding_vector_dimension = 1024
for item in tqdm(items, "Generating embeddings"):
    if item['type'] == 'text':
        item['embedding'] = generate_multimodal_embeddings(prompt=item['text'], output_embedding_length=embedding_vector_dimension)
    else:
        item['embedding'] = generate_multimodal_embeddings(image=item['image'], output_embedding_length=embedding_vector_dimension)

<!-- Section Header -->
<div id="section4" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">4. Create vector database</h2>
</div>

In this section, we will create an index using FAISS, similar to Lab 2. We will create a [FlatIndex](https://www.pinecone.io/learn/series/faiss/faiss-tutorial/#IndexFlatL2) which measures the L2 (or Euclidean) distance between all given points between our query vector, and the vectors loaded into the index.

In [ ]:
all_embeddings = np.array([item['embedding'] for item in items])

<!-- Challenge Box -->
<div style="background-color: #FEF9E7; border-left: 5px solid #F1C40F; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../mlu_utils/images/challenge.png" alt="Challenge" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #B7950B; margin-top: 0;">Challenge: Types of Index</h4>
        <p>In this lab, we will use <code>FlatIndexL2</code> as the index type for the vector database. Try using a different index and observe how the speed and the quality of results changes. You can try <code>IndexFlatIP</code>, <code>IndexHNSWFlat</code> or <code>IndexIVFFlat</code></p>
    </div>
</div>

In [ ]:
# Create FAISS Index
index = faiss.IndexFlatL2(embedding_vector_dimension)
index.reset() # Clear any pre-existing index
index.add(np.array(all_embeddings, dtype=np.float32))

<!-- Section Header -->
<div id="section5" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">5. Generate a RAG Response</h2>
</div>

In this section, we will define the function `generate_rag_response` to generate a response with a retrieval-augmented prompt. 

First, let's import the `invoke_claude_3_multimodal` function that we used in Lab 1 to generate a response to a multimodal prompt.

The following function, `generate_rag_response`, generates a prompt containing the user query, retrieved items and invokes the LLM to generate a RAG response.

In [ ]:
from mlu_utils.multimodal_utils import invoke_nova_lite_multimodal

def generate_rag_response(prompt, matched_items):
    
    # Create context
    text_context = ""
    image_context = []
    
    for item in matched_items:
        if item['type'] == 'text':
            text_context += str(item["page"]) + ". " + item['text'] + "\n"
        else:
            image_context.append(item['image'])
    
    # Only 5 images are supported by Claude3 models
    if len(image_context) > 5:
        image_context = image_context[:5]
    
    final_prompt = f"""You are a helpful assistant for question answering.
    The text context is relevant information retrieved.
    The provided image(s) are relevant information retrieved.
    
    <context>
    {text_context}
    </context>
    
    Answer the following question using the relevant context and images.
    
    <question>
    {prompt}
    </question>
    
    Answer:"""
    
    return invoke_nova_lite_multimodal(final_prompt, image_context, ['image/png' for _ in image_context])
    

<!-- Section Header -->
<div id="section6" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">6. Test RAG Workflow</h2>
</div>

Now that we have our functions ready, let's test our RAG application using a few prompts.

The steps we follow to generate a RAG response are:
1. Generate an embedding of the user query. The embedding would represent the text and the images provided in the user query.
2. Retrieve similar items from the vector database used a `nearest neighbor` search
3. Create a prompt using the user query as well as the retrieved items.
4. Generate a response using the retrieval-augmented prompt.

In [ ]:
query = "How is the scaled-dot-product attention calculated?"

query_embedding = generate_multimodal_embeddings(prompt=query,output_embedding_length=embedding_vector_dimension)
distances, result = index.search(np.array(query_embedding, dtype=np.float32).reshape(1,-1), k=5)

In [ ]:
result.flatten()

In [ ]:
matched_items = [items[index] for index in result.flatten()]

In [ ]:
response = generate_rag_response(query, matched_items)

In [ ]:
display.Markdown(response)

Nice. We have seen a few example questions and answers. Let's try asking more questions. Some example questions are given below.

<!-- Activity Box -->
<div style="background-color: #EBF5FB; border-left: 5px solid #3498DB; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../mlu_utils/images/activity.png" alt="Activity" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #2874A6; margin-top: 0;">Activity: Asking more questions about the document</h4>
        <p>So far, we have seen a few example questions. Let's use the following questions and examine the responses.</p>
        <ul>
            <li>"How long were the base and big models trained?"</li>
            <li>"Which optimizer was used when training the models?"</li>
            <li>"What is position-wise feed-forward neural network mentioned in the paper?"</li>
            <li>"What is the BLEU score of the model in english to french translation (EN-FR)?"</li>
            <li>"What is the BLEU score of the model in english to german translation (EN-DE)?"</li>
        </ul>
    </div>
</div>

<!-- Section Header -->
<div id="section7" style="border-left: 5px solid #3498DB; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2874A6;">7. Quiz Questions</h2>
</div>

Well done on completing the lab! Now, it's time for a brief knowledge assessment.

<!-- Challenge Box -->
<div style="background-color: #FEF9E7; border-left: 5px solid #F1C40F; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../mlu_utils/images/challenge.png" alt="Challenge" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #B7950B; margin-top: 0;">Challenge: Try it Yourself!</h4>
        <p>Answer the following questions to test your understanding of using prompt templates for inference.</p>
    </div>
</div>

In [ ]:
from mlu_utils.quiz_questions import lab3b_question1, lab3b_question2

lab3b_question1.display()
lab3b_question2.display()

<div style="background-color: #EBF5FB; padding: 15px; border-radius: 5px; margin: 30px 0;">
    <h3 style="color: #2874A6; border-bottom: 1px solid #85C1E9; padding-bottom: 5px;">Conclusion</h3>
    <p>In this lab, you have:</p>
    <ul>
        <li>Learned how to process PDF documents to extract text and images</li>
        <li>Generated multimodal embeddings using Amazon Nova 2 Multimodal Embeddings</li>
        <li>Created a vector database using FAISS for efficient similarity search</li>
        <li>Implemented a multimodal RAG system that can answer questions about the document</li>
        <li>Tested the RAG workflow with various queries</li>
    </ul>
    <h4 style="color: #2874A6; margin-top: 15px;">Additional Resources</h4>
    <ul>
        <li><a href="https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf">Attention Is All You Need paper</a></li>
        <li><a href="https://www.pinecone.io/learn/series/faiss/faiss-tutorial/">FAISS Tutorial</a></li>
    </ul>
</div>

<p style="padding: 10px; border: 1px solid black;">
<img src="../mlu_utils/images/MLU-NEW-logo.png" alt="drawing" width="400"/> <br/>

# Thank you!